In [ ]:


## Preparing Workspace ===============================================================



## Packages ---

import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
from datetime import datetime
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
from IPython.display import display
import traceback
import sys
# import pdb; pdb.set_trace()


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'Census'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'

path_prod = path_sp / 'Products'
path_sacsim = path_prod / 'Small Data Requests' / '2025' / 'SACSIM'



In [ ]:

geography = 'Block Groups'# 'Tracts'

path_in = path_sacsim / 'original_exports'
files = [f for f in path_in.iterdir() if f.is_file()]
files = [f for f in files if geography in str(f)]
print(files)


if geography == 'Block Groups':
    geo_id = ['County Name', 'Tract ID', 'Block Group ID']
if geography == 'Tracts':
    geo_id = ['County Name', 'Tract ID']


list_df = []

for file in tqdm(files):
    df = pd.read_excel(file)
    df = df[geo_id + ['Variable', 'Households']]
    df = df.pivot_table(index=geo_id, columns='Variable', values='Households').reset_index()
    list_df.append(df)


df = ft.reduce(lambda left, right: pd.merge(left, right, on=geo_id, how='outer'), list_df)

df = df[geo_id + [ 
         '1 person households', '2 person households', '3 person households', '4 person households', '5 or more person households', 
         'Households with householder age 0-35', 'Households with householder age 35-64', 'Households with householder age 65 plus',
         'Households with household income 0k-35k', 'Households with household income 35k-60k', 'Households with household income 60k-100k', 'Households with household income 100k-150k', 'Households with household income 150k or more',
        #  'Households with 0 workers', 'Households with 1 worker', 'Households with 2 workers', 'Households with 3 or more workers' # Include if using tracts, not available at block group level
         ]]


if geography == 'Block Groups':
    geo = 'BLOCKGROUP'
if geography == 'Tracts':
    geo = 'TRACT'

df['Geography'] = geo

df = df.set_index(['Geography'] + geo_id).reset_index()


if geography == 'Block Groups':
    cols1 = ['geography', 'county', 'tract_id', 'block_group_id']
if geography == 'Tracts':
    cols1 = ['geography', 'county', 'id']

cols = cols1 + [
        # 'num_hh_control', 'num_p_control', 
        'hh_size_1_control', 'hh_size_2_control', 'hh_size_3_control', 'hh_size_4_control', 'hh_size_5_plus_control',
        'hh_age_0_35_control', 'hh_age_35_64_control', 'hh_age_65_plus_control', 
        'hh_income_0_35_control', 'hh_income_35_60_control', 'hh_income_60_100_control', 'hh_income_100_150_control', 'hh_income_150_plus_control',
        # 'hh_workers_0_control', 'hh_workers_1_control', 'hh_workers_2_control', 'hh_workers_3_plus_control', # Include if using tracts, not available at block group level
        # 'univ_cluster_regular_control', 'univ_cluster_student_control', 
        # 'ethn_white_non_hispanic_control', 'ethn_black_non_hispanic_control', 'ethn_hispanic_control', 'ethn_asian_control', 'ethn_other_control', 
        # 'age_0_4_p_control', 'age_5_15_p_control', 'age_15_34_p_control', 'age_35_64_p_control', 'age_65_74_p_control', 'age_75_plus_p_control'
]

df.columns = cols

df



In [ ]:

file_out = path_sacsim / f'landuse_validn_tables_ss23__final_summary_{geo}.csv'
df.to_csv(file_out, index=False)
